# Adult Census Income: feature engineering

В этом notebook начинаем отдельный этап feature engineering.

Пока добавляем только согласованные признаки на основе группировки категорий:

- `is_married`;
- `native_country_group`;
- `workclass_group`.

Исходные признаки не удаляем. Новые признаки добавляются рядом с исходными, чтобы модель могла использовать и детальную категорию, и более обобщённую информацию.

## Загрузка и базовая очистка

Notebook самодостаточный: он заново загружает данные, выполняет базовую очистку и удаляет полные дубликаты.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

In [ ]:
columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income",
]

train_raw = pd.read_csv("../data/adult.data", header=None, names=columns)
test_raw = pd.read_csv("../data/adult.test", header=None, names=columns, skiprows=1)

df = pd.concat([train_raw, test_raw], ignore_index=True)

for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].str.strip()

df = df.replace("?", np.nan)
df["income"] = df["income"].str.replace(".", "", regex=False)

df_clean = df.drop_duplicates().reset_index(drop=True)

print("Rows after cleaning:", len(df_clean))
df_clean.head()

## 1. Признак `is_married`

Исходный `marital_status` содержит несколько категорий. Для модели может быть полезен более общий бинарный признак: состоит ли человек в браке сейчас.

К группе `1` относим:

- `Married-civ-spouse`;
- `Married-AF-spouse`.

Все остальные статусы относим к `0`.

In [ ]:
df_features = df_clean.copy()

married_statuses = ["Married-civ-spouse", "Married-AF-spouse"]
df_features["is_married"] = df_features["marital_status"].isin(married_statuses).astype(int)

df_features[["marital_status", "is_married"]].drop_duplicates().sort_values("marital_status")

## 2. Признак `native_country_group`

`native_country` содержит много стран, при этом основная категория — `United-States`. Редкие страны можно объединить в группу `Other`, а пропуски сохранить как `Unknown`.

Такой признак не заменяет исходную страну, а добавляет более устойчивую обобщённую информацию.

In [ ]:
df_features["native_country_group"] = np.where(
    df_features["native_country"].isna(),
    "Unknown",
    np.where(df_features["native_country"] == "United-States", "United-States", "Other"),
)

df_features["native_country_group"].value_counts()

## 3. Признак `workclass_group`

`workclass` можно сгруппировать по типу занятости:

- `Private`;
- `Government`;
- `Self-employed`;
- `Other`;
- `Unknown`.

Это снижает детализацию, но делает признак более компактным и интерпретируемым.

In [ ]:
workclass_group_map = {
    "Private": "Private",
    "Federal-gov": "Government",
    "Local-gov": "Government",
    "State-gov": "Government",
    "Self-emp-not-inc": "Self-employed",
    "Self-emp-inc": "Self-employed",
    "Without-pay": "Other",
    "Never-worked": "Other",
}

df_features["workclass_group"] = df_features["workclass"].map(workclass_group_map).fillna("Unknown")

pd.crosstab(df_features["workclass"].fillna("Unknown"), df_features["workclass_group"])

## Проверка результата

In [ ]:
new_features = ["is_married", "native_country_group", "workclass_group"]

df_features[new_features].head()

In [ ]:
print("Original shape:", df_clean.shape)
print("Shape after feature engineering:", df_features.shape)

df_features[new_features].isna().sum()

## Решение

Пока оставляем эти три engineered features:

- `is_married` — простой бинарный признак семейного статуса;
- `native_country_group` — обобщение страны происхождения до `United-States`, `Other`, `Unknown`;
- `workclass_group` — обобщение типа занятости.

Исходные признаки `marital_status`, `native_country` и `workclass` не удаляем. Позже можно сравнить качество моделей с этими признаками и без них.